# Random Forest Classifier — ML Algorithms Lab

## Hyperparameters copied carefully from the supplied cheat sheet

The cheat sheet's **Random Forest parameter grid** specifies:

- **`n_estimators`: `[50, 100, 200]`**
- **`max_depth`: `[None, 5, 10]`**
- **`criterion`: `["gini", "entropy"]`**
- **`max_features`: `["sqrt", "log2"]`**
- **`min_samples_split`: `[2, 5, 10]`**

These produce **108 parameter combinations**. The cheat sheet's syntax also lists **`random_state=<int>`**, which is fixed to `42` for reproducibility rather than searched. The grid above follows the numeric values in the cheat sheet; the syntax's optional float form of `max_features` is not added to the grid because the parameter-grid section explicitly specifies `["sqrt", "log2"]`.

Random Forest does not require feature scaling. Preprocessing is fitted inside each pipeline to avoid data leakage. The notebook includes GridSearchCV, a held-out test evaluation, multiple metrics, visualizations, feature importance, learning curve, and paired significance tests. Enter your CSV path and target column when prompted.

**Execution note:** the notebook is syntax-checked, not run on your dataset. The full grid contains 108 combinations, so runtime depends on dataset size and hardware.


In [ ]:
# ============================================================
# 1. Imports
# ============================================================
import os, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel, wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, label_binarize
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_validate, learning_curve
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, classification_report,
    ConfusionMatrixDisplay, log_loss, roc_auc_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.base import clone

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports complete. random_state =", RANDOM_STATE)

# ============================================================
# 2. Load CSV dataset
# ============================================================
csv_files = glob.glob("*.csv")
if len(csv_files) == 1:
    default_path = csv_files[0]
    print("Detected CSV:", default_path)
    path = input("Press Enter to use it, or enter another CSV path: ").strip() or default_path
elif len(csv_files) > 1:
    print("CSV files in current directory:", csv_files)
    path = input("Enter CSV path: ").strip()
else:
    path = input("Enter CSV path: ").strip()

if not os.path.exists(path):
    raise FileNotFoundError(f"Dataset not found: {path}")

df = pd.read_csv(path)
print("Dataset shape:", df.shape)
display(df.head())
print("\nData types:")
df.info()
print("\nDescriptive statistics:")
display(df.describe(include="all").T)
print("\nMissing values per column:")
display(df.isna().sum().sort_values(ascending=False).head(30))
print("Duplicate rows:", df.duplicated().sum())

# ============================================================
# 3. Select target, clean, and inspect
# ============================================================
target = input("Enter the exact target/class column name: ").strip()
if target not in df.columns:
    raise ValueError(f"Target '{target}' not found. Available columns: {list(df.columns)}")

df = df.drop_duplicates().dropna(subset=[target]).copy()
y = df[target].copy()
X = df.drop(columns=[target]).copy()

# Heuristic only: review excluded identifier-like fields before relying on results.
id_like = [c for c in X.columns if (
    c.lower() in {"id", "index", "row_id", "customerid", "customer_id", "record_id"} or
    (X[c].nunique(dropna=True) >= 0.98 * max(len(X), 1) and
     (pd.api.types.is_integer_dtype(X[c]) or X[c].dtype == "object"))
)]
if id_like:
    print("Potential identifier columns excluded; review this choice:", id_like)
    X = X.drop(columns=id_like)

if y.nunique() < 2:
    raise ValueError("Classification requires at least two target classes.")

print("Feature shape:", X.shape, "| Number of classes:", y.nunique())
display(y.value_counts().rename("class_count").to_frame())

plt.figure(figsize=(8,4))
y.astype(str).value_counts().plot(kind="bar")
plt.title("Target class distribution")
plt.xlabel("Class"); plt.ylabel("Count")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
if num_cols:
    X[num_cols].hist(figsize=(12, max(4, 2.5*((len(num_cols)+3)//4))), bins=20)
    plt.suptitle("Numeric feature distributions")
    plt.tight_layout(); plt.show()
    if len(num_cols) >= 2:
        corr = X[num_cols].corr(numeric_only=True)
        plt.figure(figsize=(9,6))
        plt.imshow(corr, aspect="auto"); plt.colorbar(label="Correlation")
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title("Numeric feature correlation matrix")
        plt.tight_layout(); plt.show()
for col in cat_cols[:6]:
    plt.figure(figsize=(8,3))
    X[col].astype(str).value_counts().head(15).plot(kind="bar")
    plt.title(f"Most frequent categories: {col}")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

# ============================================================
# 4. Split data and build leakage-safe preprocessing
# ============================================================
if int(y.value_counts().min()) < 2:
    raise ValueError("At least one class has fewer than 2 examples; stratified splitting is not reliable.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
min_train_class = int(y_train.value_counts().min())
n_splits = min(5, min_train_class)
if n_splits < 2:
    raise ValueError("Insufficient examples per class for stratified cross-validation.")

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
print("Training rows:", len(X_train), "| Test rows:", len(X_test), "| CV folds:", n_splits)

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # compatibility with older scikit-learn
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])
transformers = []
if num_cols:
    transformers.append(("num", numeric_pipe, num_cols))
if cat_cols:
    transformers.append(("cat", categorical_pipe, cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0)

# ============================================================
# 5. Random Forest model and exact cheat-sheet grid
# ============================================================
rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)
pipeline = Pipeline([("prep", preprocessor), ("model", rf)])

# EXACT GRID VALUES from the cheat sheet:
# n_estimators: [50, 100, 200]
# max_depth: [None, 5, 10]
# criterion: ["gini", "entropy"]
# max_features: ["sqrt", "log2"]
# min_samples_split: [2, 5, 10]
param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [None, 5, 10],
    "model__criterion": ["gini", "entropy"],
    "model__max_features": ["sqrt", "log2"],
    "model__min_samples_split": [2, 5, 10]
}
combos = int(np.prod([len(v) for v in param_grid.values()]))
print(f"Cheat-sheet grid combinations: {combos}")
print("random_state fixed at 42 (syntax item in the cheat sheet).")
print("Feature scaling is intentionally omitted: Random Forest does not require it.")

# ============================================================
# 6. GridSearchCV with multiple scoring metrics
# ============================================================
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted"
}
search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit="f1_macro",
    cv=cv,
    n_jobs=-1,
    error_score=np.nan,
    return_train_score=True
)
t0 = time.time()
search.fit(X_train, y_train)
elapsed = time.time() - t0
best_model = search.best_estimator_
print(f"Grid search elapsed time: {elapsed:.1f} seconds")
print("Best hyperparameters:", search.best_params_)
print("Best mean CV macro-F1:", round(search.best_score_, 5))

cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_f1_macro")
display(cv_results[[
    "rank_test_f1_macro", "mean_test_f1_macro", "std_test_f1_macro",
    "mean_test_accuracy", "mean_test_balanced_accuracy",
    "mean_test_precision_macro", "mean_test_recall_macro",
    "mean_test_f1_weighted", "params"
]].head(20))

plt.figure(figsize=(9,4))
top = cv_results.head(20).copy()
plt.plot(range(1, len(top)+1), top["mean_test_f1_macro"], marker="o")
plt.xlabel("Top candidate rank"); plt.ylabel("Mean CV macro-F1")
plt.title("Top Random Forest grid-search candidates")
plt.tight_layout(); plt.show()

# ============================================================
# 7. Held-out test-set metrics
# ============================================================
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)
classes = list(best_model.named_steps["model"].classes_)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "matthews_corrcoef": matthews_corrcoef(y_test, y_pred),
    "cohen_kappa": cohen_kappa_score(y_test, y_pred),
    "log_loss": log_loss(y_test, y_prob, labels=classes)
}
try:
    if len(classes) == 2:
        metrics["roc_auc"] = roc_auc_score(y_test, y_prob[:,1])
        metrics["average_precision"] = average_precision_score(y_test, y_prob[:,1])
    else:
        metrics["roc_auc_ovr_macro"] = roc_auc_score(
            y_test, y_prob, labels=classes, multi_class="ovr", average="macro")
        metrics["average_precision_macro"] = average_precision_score(
            label_binarize(y_test, classes=classes), y_prob, average="macro")
except Exception as e:
    print("Some probability-based metrics were unavailable:", e)

print("Held-out test metrics:")
display(pd.Series(metrics, name="test_score").to_frame())
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, xticks_rotation=45)
ax.set_title("Random Forest confusion matrix")
plt.tight_layout(); plt.show()

try:
    if len(classes) == 2:
        RocCurveDisplay.from_predictions(y_test, y_prob[:,1])
        plt.title("Random Forest ROC curve"); plt.tight_layout(); plt.show()
        PrecisionRecallDisplay.from_predictions(y_test, y_prob[:,1])
        plt.title("Random Forest precision-recall curve"); plt.tight_layout(); plt.show()
    else:
        from sklearn.metrics import roc_curve
        y_bin = label_binarize(y_test, classes=classes)
        plt.figure(figsize=(8,6))
        for i, cls in enumerate(classes):
            if len(np.unique(y_bin[:,i])) == 2:
                fpr, tpr, _ = roc_curve(y_bin[:,i], y_prob[:,i])
                plt.plot(fpr, tpr, label=f"Class {cls}")
        plt.plot([0,1], [0,1], linestyle="--")
        plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
        plt.title("Multiclass one-vs-rest ROC curves"); plt.legend()
        plt.tight_layout(); plt.show()
except Exception as e:
    print("ROC/PR plots unavailable:", e)

# ============================================================
# 8. Feature importance (after preprocessing)
# ============================================================
try:
    fitted_prep = best_model.named_steps["prep"]
    fitted_rf = best_model.named_steps["model"]
    feature_names = fitted_prep.get_feature_names_out()
    importances = pd.Series(fitted_rf.feature_importances_, index=feature_names).sort_values(ascending=False)
    display(importances.head(25).rename("importance").to_frame())
    plt.figure(figsize=(9,6))
    importances.head(20).sort_values().plot(kind="barh")
    plt.xlabel("Impurity-based importance")
    plt.title("Top Random Forest feature importances")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("Feature-importance labels could not be displayed:", e)

# ============================================================
# 9. Learning curve
# ============================================================
try:
    lc_splits = min(3, int(y_train.value_counts().min()))
    if lc_splits >= 2:
        lc_cv = StratifiedKFold(n_splits=lc_splits, shuffle=True, random_state=RANDOM_STATE)
        sizes, train_scores, val_scores = learning_curve(
            best_model, X_train, y_train, cv=lc_cv, scoring="f1_macro",
            train_sizes=np.linspace(0.3, 1.0, 4), n_jobs=-1, error_score=np.nan
        )
        plt.figure(figsize=(8,5))
        plt.plot(sizes, np.nanmean(train_scores, axis=1), marker="o", label="Train macro-F1")
        plt.plot(sizes, np.nanmean(val_scores, axis=1), marker="o", label="Validation macro-F1")
        plt.xlabel("Training examples"); plt.ylabel("Macro-F1")
        plt.title("Random Forest learning curve"); plt.legend()
        plt.tight_layout(); plt.show()
except Exception as e:
    print("Learning curve skipped:", e)

# ============================================================
# 10. Paired significance test: tuned RF vs untuned RF baseline
# ============================================================
# Use identical stratified folds for both candidates. This is a practical comparison,
# but because the tuned configuration was selected using CV on the same training data,
# the p-values should be interpreted as exploratory rather than definitive.
baseline = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1))
])
fold_scores = {"Tuned Random Forest": [], "Default Random Forest": []}
for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train), start=1):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
    tuned_fold = clone(best_model)
    base_fold = clone(baseline)
    tuned_fold.fit(X_tr, y_tr)
    base_fold.fit(X_tr, y_tr)
    tuned_pred = tuned_fold.predict(X_va)
    base_pred = base_fold.predict(X_va)
    fold_scores["Tuned Random Forest"].append(f1_score(y_va, tuned_pred, average="macro", zero_division=0))
    fold_scores["Default Random Forest"].append(f1_score(y_va, base_pred, average="macro", zero_division=0))
fold_df = pd.DataFrame(fold_scores)
print("Matched-fold macro-F1 scores:")
display(fold_df)
a = fold_df["Tuned Random Forest"].to_numpy()
b = fold_df["Default Random Forest"].to_numpy()
print("Mean difference (tuned - default):", round(float(np.mean(a-b)), 6))
if len(a) >= 2:
    print("Paired t-test:", ttest_rel(a, b, nan_policy="omit"))
    try:
        print("Wilcoxon signed-rank test:", wilcoxon(a, b, zero_method="wilcox"))
    except Exception as e:
        print("Wilcoxon test unavailable:", e)
print("Caution: CV fold scores are not fully independent; treat p-values as exploratory evidence.")

# ============================================================
# 11. Save outputs
# ============================================================
cv_summary = pd.DataFrame([{
    "model": "RandomForestClassifier",
    "best_params": str(search.best_params_),
    "best_cv_macro_f1": search.best_score_,
    "grid_search_seconds": elapsed,
    **metrics
}])
cv_summary.to_csv("random_forest_classification_summary.csv", index=False)
pd.DataFrame({"actual": y_test.reset_index(drop=True),
              "predicted": pd.Series(y_pred).reset_index(drop=True)}).to_csv(
                  "random_forest_test_predictions.csv", index=False)
cv_results.to_csv("random_forest_grid_search_results.csv", index=False)
print("Saved:")
print("- random_forest_classification_summary.csv")
print("- random_forest_test_predictions.csv")
print("- random_forest_grid_search_results.csv")
print("Notebook complete.")
